In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_txt = f.read()

print("Total number of characters:", len(raw_txt))
print(raw_txt[:100])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [ ]:
import re

txt = "Hi, I am starting to build a LLM from scratch. I am excited to embark this journey"
# This splits based on whitespace and keeps the spaces in the result 
#res = re.split(r'(\s)', txt)

# This splits based on whitespace and keeps the spaces in the result 
#res = re.split(r'\s', txt)   

#This splits based on whitespace and punctuations as well
res = re.split(r'([.,!@#%^&*_"()\--]|\s)', txt)

#Remove whitespace from the res
res = [word for word in res if word and not word.isspace()]
#Another way to rmeove whitespace from the res
#res = [word for word in res if item.strip()]

print(res)

['Hi', ',', 'I', 'am', 'starting', 'to', 'build', 'a', 'LLM', 'from', 'scratch', '.', 'I', 'am', 'excited', 'to', 'embark', 'this', 'journey']


In [13]:
preprocessed = re.split(r'([.,!:;?_!"()\']|--|\s)', raw_txt)
preprocessed = [word for word in preprocessed if word and not word.isspace()]

print(preprocessed[:100])
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--']
4690


In [19]:
#prepare a mini-vocabulary and assugn them indices
all_unique_words = sorted(set(preprocessed))
print(len(all_unique_words))

vocab = {word:index for index,word in enumerate(all_unique_words)}
for index, item in enumerate(vocab.items()):
    print(item)
    if index > 50:
        break

1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)


In [37]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {index:word for word,index in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r"([.,!:;?_!\"'()\]]|--|\s)", text)
        preprocessed = [word for word in preprocessed if word and not word.isspace()]
        ids = [self.str_to_int[word] for word in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[index] for index in ids])
        text = re.sub(r'\s+([.,!:;?_!"()\'])', r'\1', text)  # Remove space before punctuation
        return text



In [38]:
tokeniser = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
            Mrs. Gisburn said with pardonable pride"""
ids = tokeniser.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793]


In [36]:
tokeniser.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride'

In [48]:
#adding two more tokens
all_unique_words = sorted(set(preprocessed))
all_unique_words.extend(["<|endoftext|>", "<|unk|>"])
print(len(all_unique_words))

vocab = {word:index for index,word in enumerate(all_unique_words)}
for index, item in enumerate(vocab.items()):
    if index == 1130:
        print(item)

1132
('<|endoftext|>', 1130)


In [43]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {index:word for word,index in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r"([.,!:;?_!\"'()\]]|--|\s)", text)
        preprocessed = [word for word in preprocessed if word and not word.isspace()]
        preprocessed =[item if item in self.str_to_int
                       else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[word] for word in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[index] for index in ids])
        text = re.sub(r'\s+([.,!:;?_!"()\'])', r'\1', text)  # Remove space before punctuation
        return text


In [49]:
tokeniserr = SimpleTokenizerV2(vocab)
text1 = "Hello!, do you like tea?"
text2 = "In the realm of dreams, the impossible becomes possible."

text = " <|endoftext|> ".join((text1, text2))
print(text)


Hello!, do you like tea? <|endoftext|> In the realm of dreams, the impossible becomes possible.


In [51]:
tokeniserr.decode(tokeniserr.encode(text))

'<|unk|>!, do you like tea? <|endoftext|> In the <|unk|> of <|unk|>, the <|unk|> <|unk|> <|unk|>.'